# Function testing

## Goal

Lire les jeux de donnees disponibles dans `data/` et verifier qu'un batch peut etre cree avec le DataLoader du projet.

## Setup

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch

In [ ]:
def find_project_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / "data" / "jepa" / "mp.csv.gz").exists() and (path / "crystal_matrices.py").exists():
            return path
    raise FileNotFoundError("Impossible de trouver la racine du projet SUPRA-JEPA")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT, DATA_DIR

## Data

In [ ]:
data_files = sorted(path.relative_to(PROJECT_ROOT) for path in DATA_DIR.rglob("*") if path.is_file())

In [ ]:
jepa_csv_path = DATA_DIR / "jepa" / "mp.csv.gz"
raw_3dsc_path = DATA_DIR / "raw" / "3DSC_MP.csv"

jepa_csv_path, raw_3dsc_path

In [ ]:
raw_3dsc_df = pd.read_csv(raw_3dsc_path, comment="#")

print(f"Lignes: {len(raw_3dsc_df):,}")
print(f"Colonnes: {len(raw_3dsc_df.columns)}")
raw_3dsc_df[["formula_sc", "tc", "sc_class", "material_id_2"]].head()

## DataLoader

In [ ]:
from crystal_matrices import create_dataloader

dataloader = create_dataloader(jepa_csv_path, batch_size=4, shuffle=False)
batch = next(iter(dataloader))

batch_summary = {
    "material_id": batch["material_id"],
    "encoder1_atom_features_shape": tuple(batch["encoder1_atom_features"].shape),
    "encoder1_atom_mask_shape": tuple(batch["encoder1_atom_mask"].shape),
    "encoder2_atom_features_shape": tuple(batch["encoder2_atom_features"].shape),
    "encoder2_atom_mask_shape": tuple(batch["encoder2_atom_mask"].shape),
    "ef_per_atom_shape": tuple(batch["ef_per_atom"].shape),
    "num_atoms": batch["num_atoms"].tolist(),
}
batch_summary

In [ ]:
batch["material_id"]

In [ ]:
assert batch["encoder1_atom_features"].ndim == 3
assert batch["encoder1_atom_features"].shape[-1] == 103
assert batch["encoder1_atom_mask"].shape == batch["encoder1_atom_features"].shape[:2]
assert batch["encoder2_atom_features"].ndim == 3
assert batch["encoder2_atom_features"].shape[-1] == 110
assert batch["encoder2_atom_mask"].shape == batch["encoder2_atom_features"].shape[:2]
assert batch["ef_per_atom"].shape == (4,)

print("Batch DataLoader OK")

# Testing difference between datasets

Want to test if the dataset mp.csv contains all the data of 3DSC.csv 

In [27]:
import pandas as pd
import numpy as np

In [30]:
small_df = pd.read_csv("/Users/baptistecaillerie/Documents/SUPRA-JEPA/data/raw/3DSC_MP.csv", sep=",", comment="#")
mp_df = pd.read_csv("/Users/baptistecaillerie/Documents/SUPRA-JEPA/data/jepa/mp.csv.gz", comment="#")

print(f"3DSC_MP rows: {len(small_df):,}")
print(f"MP rows: {len(mp_df):,}")
print(f"MP columns: {list(mp_df.columns)}")

3DSC_MP rows: 5,773
MP rows: 92,762
MP columns: ['material_id', 'cif', 'ef_per_atom']


## Structure of `mp.csv.gz`

In [31]:
mp_column_summary = pd.DataFrame(
    {
        "column": mp_df.columns,
        "dtype": [str(dtype) for dtype in mp_df.dtypes],
        "missing_values": mp_df.isna().sum().to_numpy(),
        "missing_pct": (mp_df.isna().mean() * 100).round(2).to_numpy(),
        "unique_values": [mp_df[column].nunique(dropna=True) for column in mp_df.columns],
    }
)

mp_column_summary

,column,dtype,missing_values,missing_pct,unique_values
0,material_id,str,0,0.0,80769
1,cif,str,0,0.0,80769
2,ef_per_atom,float64,0,0.0,80688


In [32]:
mp_df[["material_id", "ef_per_atom"]].head(10)

,material_id,ef_per_atom
0,mp-23155,0.000000
1,mp-1246134,0.409911
2,mp-1182070,0.008550
3,mp-569358,0.549853
4,mp-1078637,0.051305
5,mp-567597,0.000000
6,mp-568610,0.089810
7,mp-11534,0.000000
8,mp-22912,0.064057
9,mp-1183057,0.015586


In [33]:
mp_df["ef_per_atom"].describe().to_frame()

,ef_per_atom
count,92762.000000
mean,-0.959419
std,1.220266
min,-5.154331
25%,-1.880637
50%,-0.681101
75%,-0.192126
max,5.456221


In [34]:
from pymatgen.core import Structure

example_row = mp_df.iloc[0]
example_structure = Structure.from_str(example_row["cif"], fmt="cif")

example_structure_summary = pd.DataFrame(
    [
        {
            "material_id": example_row["material_id"],
            "formula": example_structure.composition.reduced_formula,
            "num_sites": len(example_structure),
            "lattice_a": example_structure.lattice.a,
            "lattice_b": example_structure.lattice.b,
            "lattice_c": example_structure.lattice.c,
            "alpha": example_structure.lattice.alpha,
            "beta": example_structure.lattice.beta,
            "gamma": example_structure.lattice.gamma,
            "ef_per_atom": example_row["ef_per_atom"],
        }
    ]
)

example_structure_summary

/Users/baptistecaillerie/.local/share/uv/python/cpython-3.12.10-macos-aarch64-none/lib/python3.12/functools.py:998: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  val = self.func(instance)


,material_id,formula,num_sites,lattice_a,lattice_b,lattice_c,alpha,beta,gamma,ef_per_atom
0,mp-23155,Ar,1,3.988628,3.988628,3.988628,60.0,60.0,60.0,0.0


In [35]:
small_df.head()

,formula_sc,formula_similarity,totreldiff,formula_frac,correct_formula_frac,formula_2,orig_formula_cif,tc,sc_class,sc_class_unique_sc,...,monoclinic,orthorhombic,tetragonal,triclinic,trigonal,primitive,base-centered,body-centered,face-centered,weight
0,Ag0.02Ge2Pd1.98Sr1,2,0.008000,1.0,True,Ag0.02Ge2Pd1.98Sr1,Ge2Pd2Sr1,2.64,Other,True,...,0,0,7,0,0,0,0,1,0,1.0
1,Ag0.15Sn0.85Te1,3,0.150000,1.0,True,Ag0.15Sn0.85Te1,Sn1Te1,2.15,Other,True,...,0,0,0,0,0,0,0,0,1,1.0
2,Ag0.1Ge2Pd1.9Sr1,2,0.040000,1.0,True,Ag0.1Ge2Pd1.9Sr1,Ge2Pd2Sr1,2.62,Other,True,...,0,0,7,0,0,0,0,1,0,1.0
3,Ag0.1In0.9Te1,3,0.100000,1.0,True,Ag0.1In0.9Te1,In1Te1,1.20,Other,True,...,0,0,0,0,0,0,0,0,1,1.0
4,Ag0.2Ba1Si1.8,3,0.133333,4.0,False,Ag0.8Ba4Si7.2,Ba4Si8,3.20,Other,True,...,0,0,0,0,0,1,0,0,0,1.0


In [37]:
mp_df.head()

,material_id,cif,ef_per_atom
0,mp-23155,# generated using pymatgen\ndata_Ar\n_symmetry...,0.000000
1,mp-1246134,# generated using pymatgen\ndata_Ni\n_symmetry...,0.409911
2,mp-1182070,# generated using pymatgen\ndata_Bi\n_symmetry...,0.008550
3,mp-569358,# generated using pymatgen\ndata_Bi\n_symmetry...,0.549853
4,mp-1078637,# generated using pymatgen\ndata_Bi\n_symmetry...,0.051305
